In [1]:
import pathlib
import sys
import xarray as xr

import intake
import intake_esm

LIBDIR = pathlib.Path("/glade/u/home/bbuchovecky/coup_ppe/src/coup_ppe/").resolve()
sys.path.insert(0, str(LIBDIR))
import access.catalog
import access.load
import parallel.daskhelper
import access.inspect
import metadata.conventions
import metadata.members

In [8]:
metadata.members.get_canonical_member_ids(["4", "fff,min", "d_max"], "fhist")

['3', '4', '7']

In [2]:
access.load.load_ppe(
    ensemble = "pisom",
    varname = ["FSNT", "FLNT"],
    component = "atm",
    frequency = "month_1",
    member = None,
    catalog_path = "/glade/u/home/bbuchovecky/coup_ppe/dev/pisom_timeseries_catalog_mem.json",
    derived_registry = None,
    xarray_open_kwargs = None,
    chunks = None,
    preprocess = lambda ds: ds.sel(time=slice("0050-01", "0184-12")),
)

<xarray.Dataset> Size: 358MB
Dimensions:    (member: 2, time: 1620, lat: 96, lon: 144, nbnd: 2)
Coordinates:
  * member     (member) object 16B '6' '8'
  * time       (time) object 13kB 0050-01-16 12:00:00 ... 0184-12-16 12:00:00
  * lat        (lat) float64 768B -90.0 -88.11 -86.21 ... 86.21 88.11 90.0
  * lon        (lon) float64 1kB 0.0 2.5 5.0 7.5 ... 350.0 352.5 355.0 357.5
    time_bnds  (time, nbnd) object 26kB dask.array<chunksize=(1620, 2), meta=np.ndarray>
Dimensions without coordinates: nbnd
Data variables:
    FSNT       (member, time, lat, lon) float32 179MB dask.array<chunksize=(1, 1620, 96, 144), meta=np.ndarray>
    FLNT       (member, time, lat, lon) float32 179MB dask.array<chunksize=(1, 1620, 96, 144), meta=np.ndarray>
Attributes: (12/21)
    Conventions:                          CF-1.0
    source:                               CAM
    logname:                              czarakas
    host:                                 cheyenne5
    initial_file:                         /glade/p/cesmdata/cseg/inputdata/ce...
    topography_file:                      /glade/p/cesmdata/cseg/inputdata/at...
    ...                                   ...
    intake_esm_attrs:frequency:           month_1
    intake_esm_attrs:time_steps:          1680
    intake_esm_attrs:start:               0049-01-16 12:00:00
    intake_esm_attrs:end:                 0188-12-16 12:00:00
    intake_esm_attrs:_data_format_:       netcdf
    intake_esm_dataset_key:               FSNT.atm.h1.month_1

In [3]:
outpath = pathlib.Path("/glade/u/home/bbuchovecky/coup_ppe/scripts").resolve()

In [7]:
pathlib.Path(outpath.parts[1])

PosixPath('glade')

In [3]:
cat = intake.open_esm_datastore("/glade/u/home/bbuchovecky/coup_ppe/dev/pisom_timeseries_catalog_mem.json")

In [5]:
cat.df

,member,version,component,stream,variable,path,variable_long_name,frequency,time_steps,start,end
0,6,2,atm,h1,FLNT,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,Net longwave flux at top of model,month_1,1680,0049-01-16 12:00:00,0188-12-16 12:00:00
1,6,2,atm,h1,FSNT,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,Net solar flux at top of model,month_1,1680,0049-01-16 12:00:00,0188-12-16 12:00:00
2,8,2,atm,h1,FLNT,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,Net longwave flux at top of model,month_1,1680,0049-01-16 12:00:00,0188-12-16 12:00:00
3,8,2,atm,h1,FSNT,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,Net solar flux at top of model,month_1,1680,0049-01-16 12:00:00,0188-12-16 12:00:00


In [15]:
ensemble = "pisom"
varname = "FLNT"
component = "atm"
frequency = "month_1"
member = "zsno,max"
catalog_path = "/glade/u/home/bbuchovecky/coup_ppe/dev/pisom_timeseries_catalog_mem.json"
derived_registry = None
xarray_open_kwargs = None
chunks = None
preprocess = None

# Ensure varname and member are lists
vars_to_load = [varname] if isinstance(varname, str) else varname
mems_to_load = [member] if isinstance(member, (int, str)) else member

# TODO:
# implement logic to find default catalog_path if not provided, put default paths in metadata
# should it look for timeseries or history catalog?
if not catalog_path:
    # cat_path = access.paths.DEFAULT_CATALOG_PATH[ensemble]
    raise NotImplementedError
else:
    cat_path = pathlib.Path(catalog_path)
    assert cat_path.suffix == '.json', f"File must be a JSON file, got {cat_path.suffix}"


# TODO: implement DerivedVariableRegistry
if not derived_registry:
    # derived_registry = derived.derived_variable_registry.DVR
    pass
else:
    assert isinstance(derived_registry, intake_esm.derived.DerivedVariableRegistry)

# Open catalog
cat = intake.open_esm_datastore(cat_path, registry=derived_registry)

# Parse the members
if mems_to_load is not None:
    members_query = metadata.members.get_canonical_member_ids(mems_to_load, ensemble)
else:
    members_query = None

# Search for the requested data
query = dict(
    variable=vars_to_load,
    component=component,
    frequency=frequency,
    member=members_query
)
cat_subset = cat.search(**query)

# Set up xarray kwargs
open_kwargs = xarray_open_kwargs or {}
if "use_cftime" not in open_kwargs:
    open_kwargs["use_cftime"] = True
if chunks:
    open_kwargs["chunks"] = chunks

# Load each variable separately to avoid coordinate conflicts
datasets = []
for var in vars_to_load:
    var_cat = cat_subset.search(variable=var)

    ds_dict = var_cat.to_dataset_dict(
        xarray_open_kwargs=open_kwargs,
        xarray_combine_by_coords_kwargs={"join": "outer"},
        preprocess=preprocess,
        threaded=True,
        progressbar=False,
    )

    # Combine all keys for this variable
    var_ds = list(ds_dict.values())
    if len(var_ds) == 1:
        datasets.append(var_ds[0])
    else:
        datasets.append(xr.merge(var_ds))

# Merge all variables together
result = xr.merge(datasets)

In [16]:
result

<xarray.Dataset> Size: 186MB
Dimensions:    (time: 1680, nbnd: 2, member: 1, lat: 96, lon: 144)
Coordinates:
  * time       (time) object 13kB 0049-01-16 12:00:00 ... 0188-12-16 12:00:00
  * member     (member) object 8B '6'
  * lat        (lat) float64 768B -90.0 -88.11 -86.21 ... 86.21 88.11 90.0
  * lon        (lon) float64 1kB 0.0 2.5 5.0 7.5 ... 350.0 352.5 355.0 357.5
Dimensions without coordinates: nbnd
Data variables:
    time_bnds  (time, nbnd) object 27kB dask.array<chunksize=(1680, 2), meta=np.ndarray>
    FLNT       (member, time, lat, lon) float32 93MB dask.array<chunksize=(1, 1680, 96, 144), meta=np.ndarray>
    var        (time, lat, lon) float32 93MB dask.array<chunksize=(1680, 96, 144), meta=np.ndarray>
Attributes: (12/25)
    Conventions:                          CF-1.0
    source:                               CAM
    case:                                 COUP0006_PI_SOM_v02
    logname:                              czarakas
    host:                                 cheyenne5
    initial_file:                         /glade/p/cesmdata/cseg/inputdata/ce...
    ...                                   ...
    intake_esm_attrs:frequency:           month_1
    intake_esm_attrs:time_steps:          1680
    intake_esm_attrs:start:               0049-01-16 12:00:00
    intake_esm_attrs:end:                 0188-12-16 12:00:00
    intake_esm_attrs:_data_format_:       netcdf
    intake_esm_dataset_key:               FLNT.atm.h1.month_1

In [11]:
cat.search(
    variable=["TS", "TREFHT"],
    frequency="month_1",
).df

,member,version,component,stream,variable,path,frequency


In [ ]:
# client, cluster = parallel.daskhelper.create_dask_cluster("UWAS0155", 10)

In [ ]:
# pisom_hist = "COUP0001_PI_SOM_v02.cam.h1.0059-04.nc"
# fhist_hist = "f.e21.FHIST_BGC.f19_f19_mg17.historical.coupPPE.001.cam.h2.1982-01-01-00000.nc"

# pisom_proc = "COUP0000_PI_SOM.cam.h0.timeseries.AODDUST.nc"

In [ ]:
# ds = xr.open_dataset("/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0000_PI_SOM/atm/proc/tseries/COUP0000_PI_SOM.cam.h0.timeseries.AODDUST.nc")

In [ ]:
# access.catalog.build_catalog(
#     ensemble="pisom",
#     kind="timeseries",
#     rootpath="/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0000_PI_SOM/atm/proc/tseries",
#     outpath=".",
#     tag="_atm",
#     depth=1)

In [ ]:
rootpath = [
    # "/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0006_PI_SOM_v02/atm/proc/tseries/COUP0006_PI_SOM_v02.cam.h1.timeseries.FLNT.nc",
    # "/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0008_PI_SOM_v02/atm/proc/tseries/COUP0008_PI_SOM_v02.cam.h1.timeseries.FLNT.nc",
    # "/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0006_PI_SOM_v02/atm/proc/tseries/COUP0006_PI_SOM_v02.cam.h1.timeseries.FSNT.nc",
    # "/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0008_PI_SOM_v02/atm/proc/tseries/COUP0008_PI_SOM_v02.cam.h1.timeseries.FSNT.nc",
    "/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0006_PI_SOM_v02/atm/hist/COUP0006_PI_SOM_v02.cam.h0.0049-01.nc",
    "/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0006_PI_SOM_v02/atm/hist/COUP0006_PI_SOM_v02.cam.h0.0049-02.nc",
    "/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0008_PI_SOM_v02/atm/hist/COUP0008_PI_SOM_v02.cam.h0.0049-01.nc",
    "/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0008_PI_SOM_v02/atm/hist/COUP0008_PI_SOM_v02.cam.h0.0049-02.nc",
]

access.catalog.build_catalog(
    ensemble="pisom",
    kind="history",
    rootpath=rootpath,
    outpath=".",
    tag="_mem",
    depth=0)

Successfully wrote ESM catalog json file to: file:///glade/u/home/bbuchovecky/coup_ppe/dev/pisom_history_catalog_mem.json


'/glade/u/home/bbuchovecky/coup_ppe/dev/pisom_history_catalog_mem.json'

In [27]:
cat = intake.open_esm_datastore(
    "pisom_history_catalog_mem.json",
)

In [28]:
cat.df

,member,version,component,stream,date,path,frequency,time_steps,start,end,variable
0,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,gw
1,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,hyam
2,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,hybm
3,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,P0
4,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,hyai
...,...,...,...,...,...,...,...,...,...,...,...
1439,8,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,soa_a2_SRF
1440,8,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,soa_c1
1441,8,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,soa_c1SFWET
1442,8,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,soa_c2


In [29]:
query = dict(stream="h0", variable=["TS", "TREFHT"])
cat_subset = cat.search(**query)
cat_subset.df

,member,version,component,stream,date,path,frequency,time_steps,start,end,variable
0,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,TREFHT
1,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,TS
2,6,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,TREFHT
3,6,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,TS
4,8,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,TREFHT
5,8,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,TS
6,8,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,TREFHT
7,8,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,TS


In [34]:
def subset_time(ds):
    return ds.sel(time=slice("0050-01", "0184-12"))

vars = ["TS", "TREFHT"]

query = dict(stream="h0", variable=vars)
cat_subset = cat.search(**query)
cat_subset.df

data_dict = cat_subset.to_dataset_dict(
    # preprocess=lambda ds: ds[vars],
    # xarray_open_kwargs={'chunks': {'time': 365}},
    threaded=False,
    xarray_open_kwargs={
        # "drop_variables": ["var"],
        "use_cftime": True},
    # xarray_combine_by_coords_kwargs={
    #     "join": "outer",
    #     "compat": "override"},
    progressbar=False,
)

/glade/u/home/bbuchovecky/miniforge3/envs/coup-ppe/lib/python3.14/site-packages/intake_esm/source.py:298: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  self._ds = xr.combine_by_coords(
/glade/u/home/bbuchovecky/miniforge3/envs/coup-ppe/lib/python3.14/site-packages/intake_esm/source.py:298: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
 

ESMDataSourceError: Failed to load dataset with key='atm.h0.month_1'
                 You can use `cat['atm.h0.month_1'].df` to inspect the assets/files for this key.
                 

In [35]:
data_dict

{'atm.h0.month_1': <xarray.Dataset> Size: 500MB
 Dimensions:           (time: 2, lat: 96, lon: 144, lev: 32, ilev: 33, nbnd: 2)
 Coordinates: (12/367)
   * time              (time) object 16B 0049-02-01 00:00:00 0049-03-01 00:00:00
   * lat               (lat) float64 768B -90.0 -88.11 -86.21 ... 88.11 90.0
   * lon               (lon) float64 1kB 0.0 2.5 5.0 7.5 ... 352.5 355.0 357.5
   * lev               (lev) float64 256B 3.643 7.595 14.36 ... 957.5 976.3 992.6
   * ilev              (ilev) float64 264B 2.255 5.032 10.16 ... 985.1 1e+03
     gw                (lat) float64 768B dask.array<chunksize=(96,), meta=np.ndarray>
     ...                ...
     soa_a2SFWET       (time, lat, lon) float32 111kB dask.array<chunksize=(1, 96, 144), meta=np.ndarray>
     soa_a2_SRF        (time, lat, lon) float32 111kB dask.array<chunksize=(1, 96, 144), meta=np.ndarray>
     soa_c1            (time, lev, lat, lon) float32 4MB dask.array<chunksize=(1, 32, 96, 144), meta=np.ndarray>
     soa_c1SF

In [18]:
cat_subset['atm.h0.month_1'].df

,member,version,component,stream,date,path,frequency,time_steps,start,end,variable,_data_format_
0,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,TREFHT,netcdf
1,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,TS,netcdf
2,6,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,TREFHT,netcdf
3,6,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,TS,netcdf
4,8,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,TREFHT,netcdf
5,8,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,TS,netcdf
6,8,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,TREFHT,netcdf
7,8,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,TS,netcdf


In [ ]:
# client.shutdown()
# del cluster
# del client
# !rm ./dask-worker.e*
# !rm ./dask-worker.o*

In [ ]:
import pathlib
import joblib

import xarray as xr
import intake
from ecgtools import Builder
from ecgtools.builder import INVALID_ASSET, TRACEBACK

LIBDIR = pathlib.Path("/glade/u/home/bbuchovecky/coup_ppe/src/coup_ppe/").resolve()
sys.path.insert(0, str(LIBDIR))
import access.catalog
import parallel.daskhelper
import access.inspect
import metadata.conventions

def build_catalog(
        ensemble: str,
        kind: str,
        rootpath: list[str],
        outpath: str,
        tag: str = "",
        depth: int | None = None,
    ) -> str:
    """
    Build an intake-esm catalog for a CESM PPE.
    
    Parameters
    ----------
    ensemble : str
        Name of the ensemble
    kind : str
        Type of catalog: "history" or "timeseries"
    rootpath : list[str]
        Root paths to search for data files
    outpath : str, optional
        Output directory path
    depth : int, optional
        Directory depth to search
        
    Returns
    -------
    str
        Path to the saved catalog JSON file
    """
    assert kind in ["history", "timeseries"], "kind must be \"history\" or \"timeseries\""

    # Get the correct parsing function
    if ensemble == "pisom":
        if kind == "history":
            parser = access.catalog.parse_pisom_history
        else:
            parser = access.catalog.parse_pisom_timeseries
    elif ensemble == "fhist":
        if kind == "history":
            parser = access.catalog.parse_fhist_history
        else:
            raise NotImplementedError
    elif ensemble == "fssp370":
        raise NotImplementedError
    else:
        raise ValueError("not a valid ensemble name (either pisom, fhist, or fssp370)")

    # Configuration for different catalog types
    config = {
        "history": {
            "depth": depth or 5,
            "exclude_patterns": ["*/proc/*", "*/rest/*"],
            "groupby_attrs": ["variable", "component", "stream", "frequency"],
            "aggregation": 
            [
                {
                    "type": "join_existing",
                    "attribute_name": "date",
                    "options": {
                        "dim": "time",
                        "coords": "minimal",
                        "compat": "override",
                    },
                },
                # {
                #     "type": "join_new",
                #     "attribute_name": "member",
                #     "options": {
                #         "dim": "member",
                #         "coords": "minimal",
                #     },
                # }
            ],
        },
        "timeseries": {
            "depth": depth or 2,
            "exclude_patterns": ["*/hist/*", "*/rest/*"],
            "groupby_attrs": ["variable", "component", "stream", "frequency"],
            "aggregation":
            [
                {
                    "type": "union",
                    "attribute_name": "variable",
                },
                {
                    "type": "join_new",
                    "attribute_name": "member",
                    "options": {
                        "dim": "member",
                        "coords": "minimal",
                    },
                }
            ],
        }
    }

    cfg = config[kind]

     # Convert to pathlib.Path and ensure directory exists
    outpath = pathlib.Path(outpath).resolve()

    # Create the catalog builder
    cat_builder = Builder(
        paths=rootpath,
        depth=cfg["depth"],
        exclude_patterns=cfg["exclude_patterns"],
        joblib_parallel_kwargs={"n_jobs": -1},
    )

    # Build the catalog
    if parallel.daskhelper.is_dask_available():
        with joblib.parallel_backend('dask', scatter=[parser]):
            cat_builder = cat_builder.build(parsing_func=parser)
    else:
        cat_builder = cat_builder.build(parsing_func=parser)

    # Clean the dataframe
    cat_builder.clean_dataframe()

    # For history files, expand variable lists into separate rows
    if kind == "history":
        df = cat_builder.df
        df = df.explode("variable").reset_index(drop=True)
        cat_builder.df = df

    # Save the catalog with proper time concatenation settings
    catalog_path = f"{ensemble}_{kind}_catalog{tag}"
    cat_builder.save(
        name=catalog_path,
        directory=str(outpath),
        path_column_name="path", 
        variable_column_name="variable",
        data_format="netcdf",
        groupby_attrs=cfg["groupby_attrs"],
        aggregations=cfg["aggregation"],
    )

    return str((outpath / catalog_path).with_suffix('.json'))

In [57]:
rootpath = [
    # "/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0006_PI_SOM_v02/atm/proc/tseries/COUP0006_PI_SOM_v02.cam.h1.timeseries.FLNT.nc",
    # "/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0008_PI_SOM_v02/atm/proc/tseries/COUP0008_PI_SOM_v02.cam.h1.timeseries.FLNT.nc",
    # "/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0006_PI_SOM_v02/atm/proc/tseries/COUP0006_PI_SOM_v02.cam.h1.timeseries.FSNT.nc",
    # "/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0008_PI_SOM_v02/atm/proc/tseries/COUP0008_PI_SOM_v02.cam.h1.timeseries.FSNT.nc",
    "/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0006_PI_SOM_v02/atm/hist/COUP0006_PI_SOM_v02.cam.h0.0049-01.nc",
    "/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0006_PI_SOM_v02/atm/hist/COUP0006_PI_SOM_v02.cam.h0.0049-02.nc",
    "/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0008_PI_SOM_v02/atm/hist/COUP0008_PI_SOM_v02.cam.h0.0049-01.nc",
    "/glade/campaign/cgd/tss/czarakas/CoupledPPE/coupled_simulations/COUP0008_PI_SOM_v02/atm/hist/COUP0008_PI_SOM_v02.cam.h0.0049-02.nc",
]

build_catalog(
    ensemble="pisom",
    kind="history",
    rootpath=rootpath,
    outpath=".",
    tag="_mem",
    depth=0)

cat = intake.open_esm_datastore("pisom_history_catalog_mem.json")
cat.df

Successfully wrote ESM catalog json file to: file:///glade/u/home/bbuchovecky/coup_ppe/dev/pisom_history_catalog_mem.json


,member,version,component,stream,date,path,frequency,time_steps,start,end,variable
0,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,gw
1,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,hyam
2,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,hybm
3,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,P0
4,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,hyai
...,...,...,...,...,...,...,...,...,...,...,...
1439,8,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,soa_a2_SRF
1440,8,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,soa_c1
1441,8,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,soa_c1SFWET
1442,8,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,soa_c2


In [58]:
# def subset_time(ds):
#     return ds.sel(time=slice("0050-01", "0184-12"))

# vars = ["TS", "TREFHT"]
vars = ["TS", "TREFHT"]

query = dict(stream="h0", variable=vars)
cat_subset = cat.search(**query)
cat_subset.df

,member,version,component,stream,date,path,frequency,time_steps,start,end,variable
0,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,TREFHT
1,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,TS
2,6,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,TREFHT
3,6,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,TS
4,8,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,TREFHT
5,8,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,TS
6,8,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,TREFHT
7,8,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,TS


In [61]:
xr.set_options(use_new_combine_kwarg_defaults=True)

data_dict = cat_subset.to_dataset_dict(
    preprocess=lambda ds: ds[vars],
    # xarray_open_kwargs={'chunks': {'time': 365}},
    threaded=False,
    xarray_open_kwargs={
        # "drop_variables": ["var"],
        "use_cftime": True},
    xarray_combine_by_coords_kwargs={
        "join": "outer",
        },
    progressbar=False,
)

data_dict["TS.atm.h0.month_1"]

<xarray.Dataset> Size: 334kB
Dimensions:  (member: 2, time: 2, lat: 96, lon: 144)
Coordinates:
  * member   (member) object 16B '6' '8'
  * time     (time) object 16B 0049-02-01 00:00:00 0049-03-01 00:00:00
  * lat      (lat) float64 768B -90.0 -88.11 -86.21 -84.32 ... 86.21 88.11 90.0
  * lon      (lon) float64 1kB 0.0 2.5 5.0 7.5 10.0 ... 350.0 352.5 355.0 357.5
    TREFHT   (time, lat, lon) float32 111kB dask.array<chunksize=(1, 96, 144), meta=np.ndarray>
Data variables:
    TS       (member, time, lat, lon) float32 221kB dask.array<chunksize=(1, 1, 96, 144), meta=np.ndarray>
Attributes: (12/17)
    Conventions:                     CF-1.0
    source:                          CAM
    logname:                         czarakas
    host:                            cheyenne5
    initial_file:                    /glade/p/cesmdata/cseg/inputdata/cesm2_i...
    topography_file:                 /glade/p/cesmdata/cseg/inputdata/atm/cam...
    ...                              ...
    intake_esm_attrs:stream:         h0
    intake_esm_attrs:frequency:      month_1
    intake_esm_attrs:time_steps:     1
    intake_esm_attrs:variable:       TS
    intake_esm_attrs:_data_format_:  netcdf
    intake_esm_dataset_key:          TS.atm.h0.month_1

In [78]:
cat_subset.df

,member,version,component,stream,date,path,frequency,time_steps,start,end,variable
0,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,TREFHT
1,6,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,TS
2,6,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,TREFHT
3,6,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,TS
4,8,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,TREFHT
5,8,2,atm,h0,0049-01,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-02-01 00:00:00,0049-02-01 00:00:00,TS
6,8,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,TREFHT
7,8,2,atm,h0,0049-02,/glade/campaign/cgd/tss/czarakas/CoupledPPE/co...,month_1,1,0049-03-01 00:00:00,0049-03-01 00:00:00,TS
